In [ ]:

import pandas as pd
# regular expressions -> re
import re
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
#Converts text messages into numerical features TF-IDF stands for Term Frequency–Inverse Document Frequency.
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay,
    classification_report, accuracy_score
)

# 1. Load labeled spam dataset
df = pd.read_csv("spam.csv")

# Support common SMS Spam Collection column formats
if "v1" in df.columns and "v2" in df.columns:
    df = df.rename(columns={"v1": "label", "v2": "message"})
elif "label" in df.columns and "message" in df.columns:
    pass
else:
  #selects the first two columns
    df = df.iloc[:, :2]
    df.columns = ["label", "message"]

df = df[["label", "message"]].dropna()
df["label"] = df["label"].str.lower().str.strip()
df = df[df["label"].isin(["spam", "ham"])]

print("First 5 messages:")
print(df.head())

print("\nClass Distribution:")
print(df["label"].value_counts())

# 2. Clean text
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()  #remove whitespaces
    return text

df["message"] = df["message"].apply(clean_text)

# Separate messages and labels
X = df["message"]
y = df["label"]

# 3. Split data before fitting the vectorizer
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 4. TF-IDF + Multinomial Naive Bayes(Multinomial refers to modelling the occurrence or frequency of words in documents)
# The vectorizer is fitted on training messages only.

model = Pipeline([

    #TfidfVectorizer() converts the cleaned messages into numerical feature vectors.
    #stop_words="english" removes common English words, such as "the" and "is"
    ("tfidf", TfidfVectorizer(stop_words="english")),

    #uses the numeric values assigned by tfidf to Estimates the spam and ham probabilities
    ("classifier", MultinomialNB())
])

model.fit(X_train, y_train)

# 5. Predict test messages
y_pred = model.predict(X_test)

print("\nActual vs Predicted:")
print(pd.DataFrame({
    "Actual": y_test.values[:10],
    "Predicted": y_pred[:10]
}).to_string(index=False))

# 6. Evaluate the model
print("\nModel Evaluation:")
print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("Precision:", round(precision_score(
    y_test, y_pred, pos_label="spam", zero_division=0
), 4))
print("Recall:", round(recall_score(
    y_test, y_pred, pos_label="spam", zero_division=0
), 4))
print("F1 Score:", round(f1_score(
    y_test, y_pred, pos_label="spam", zero_division=0
), 4))

print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

# 7. Display confusion matrix
labels = ["ham", "spam"]
cm = confusion_matrix(y_test, y_pred, labels=labels)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Ham", "Spam"]
)
disp.plot(cmap="Blues")
plt.title("Spam Detection Confusion Matrix")
plt.show()

# 8. Predict a new message entered by the user
new_message = input("\nEnter a message to classify: ")
prediction = model.predict([clean_text(new_message)])[0]

print("Predicted Class:", prediction.upper())
